# Task 2 – Data Engineering
**Module:** 7006SCN – Machine Learning and Big Data
**Focus:** Feature Engineering, PySpark Pipeline Construction


In [1]:
# ============================================================
# TASK 2: Data Engineering & Pipeline Construction
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, hour, dayofweek, month, year,
    unix_timestamp, isnull, isnan, count, lit,
    log1p, percentile_approx
)
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler, StandardScaler,
    MinMaxScaler, Bucketizer
)

spark = SparkSession.builder \
    .appName('7006SCN_Task2_DataEngineering') \
    .config('spark.executor.memory', '4g') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print('SparkSession initialised')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 13:49:02 WARN Utils: Your hostname, Naveenrajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.34 instead (on interface en0)
26/06/29 13:49:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 13:49:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 13:49:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/29 13:49:03 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


SparkSession initialised


In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = spark.read.parquet('./data/yellow_tripdata_2023-01.parquet')
print(f'Loaded {df.count():,} rows, {len(df.columns)} columns')

# Show partition count (Big Data evidence)
print(f'Number of partitions: {df.rdd.getNumPartitions()}')
df.show(5)

Loaded 3,066,766 rows, 19 columns
Number of partitions: 12
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:32:10|  2023-01-01 00:40:36|            1.0|         0.97|       1.0|                 N|         161|  

In [3]:
# ============================================================
# STEP 1: DATA QUALITY ASSESSMENT
# ============================================================

print('=== Missing Value Analysis ===')
total = df.count()
for c in df.columns:
    null_n = df.filter(isnull(col(c))).count()
    pct = round((null_n / total) * 100, 2)
    if null_n > 0:
        print(f'  {c}: {null_n:,} nulls ({pct}%)')

print('\n=== Duplicate Check ===')
dupes = df.count() - df.dropDuplicates().count()
print(f'Duplicate rows: {dupes:,}')

=== Missing Value Analysis ===
  passenger_count: 71,743 nulls (2.34%)
  RatecodeID: 71,743 nulls (2.34%)
  store_and_fwd_flag: 71,743 nulls (2.34%)
  congestion_surcharge: 71,743 nulls (2.34%)
  airport_fee: 71,743 nulls (2.34%)

=== Duplicate Check ===


[Stage 70:===================================================>    (11 + 1) / 12]

Duplicate rows: 0


In [4]:
# ============================================================
# STEP 2: DATA CLEANING
# Remove invalid records (negative fares, zero distance, etc.)
# ============================================================

df_clean = df \
    .filter(col('fare_amount') > 0) \
    .filter(col('trip_distance') > 0) \
    .filter(col('passenger_count') > 0) \
    .filter(col('total_amount') > 0) \
    .filter(col('tip_amount') >= 0) \
    .filter(col('tpep_pickup_datetime').isNotNull()) \
    .filter(col('tpep_dropoff_datetime').isNotNull()) \
    .dropDuplicates()

print(f'Rows after cleaning: {df_clean.count():,}')
print(f'Rows removed: {df.count() - df_clean.count():,}')

Rows after cleaning: 2,884,228


[Stage 85:===================================================>    (11 + 1) / 12]

Rows removed: 182,538


In [5]:
# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================

# Datetime feature extraction
df_feat = df_clean \
    .withColumn('pickup_hour', hour(col('tpep_pickup_datetime'))) \
    .withColumn('pickup_day', dayofweek(col('tpep_pickup_datetime'))) \
    .withColumn('pickup_month', month(col('tpep_pickup_datetime'))) \
    .withColumn('is_weekend', when(dayofweek(col('tpep_pickup_datetime')).isin([1,7]), 1).otherwise(0)) \
    .withColumn('is_rush_hour', when(
        (hour(col('tpep_pickup_datetime')).between(7, 9)) |
        (hour(col('tpep_pickup_datetime')).between(17, 19)), 1).otherwise(0)) \
    .withColumn('trip_duration_min',
        (unix_timestamp('tpep_dropoff_datetime') - unix_timestamp('tpep_pickup_datetime')) / 60) \
    .withColumn('speed_mph',
        when(col('trip_duration_min') > 0,
             col('trip_distance') / (col('trip_duration_min') / 60)).otherwise(0)) \
    .withColumn('fare_per_mile',
        when(col('trip_distance') > 0,
             col('fare_amount') / col('trip_distance')).otherwise(0))

# TARGET VARIABLE: Binary tip classification
# High tip = tip_amount > 20% of fare_amount
df_feat = df_feat.withColumn('high_tip',
    when(col('tip_amount') > col('fare_amount') * 0.2, 1).otherwise(0))

print('Feature engineering complete')
df_feat.select(
    'pickup_hour', 'pickup_day', 'is_weekend',
    'is_rush_hour', 'trip_duration_min', 'speed_mph',
    'fare_per_mile', 'high_tip'
).show(5)

Feature engineering complete


[Stage 91:===================================================>    (11 + 1) / 12]

+-----------+----------+----------+------------+------------------+------------------+------------------+--------+
|pickup_hour|pickup_day|is_weekend|is_rush_hour| trip_duration_min|         speed_mph|     fare_per_mile|high_tip|
+-----------+----------+----------+------------+------------------+------------------+------------------+--------+
|          0|         1|         1|           0|37.916666666666664|  9.39956043956044|6.2794612794612785|       1|
|          0|         1|         1|           0|              14.6|11.958904109589042| 5.841924398625429|       1|
|          0|         1|         1|           0|              9.25| 11.74054054054054| 6.298342541436464|       0|
|          0|         1|         1|           0| 36.81666666666667|4.1557265731100035|11.607843137254903|       1|
|          0|         1|         1|           0|14.666666666666666|11.331818181818182| 6.137184115523466|       0|
+-----------+----------+----------+------------+------------------+-------------

In [6]:
# ============================================================
# STEP 4: ENCODING CATEGORICAL FEATURES
# StringIndexer for payment_type and VendorID
# ============================================================

# Convert to string for StringIndexer
df_feat = df_feat \
    .withColumn('payment_type_str', col('payment_type').cast('string')) \
    .withColumn('vendor_str', col('VendorID').cast('string'))

# StringIndexer for categorical columns
payment_indexer = StringIndexer(
    inputCol='payment_type_str',
    outputCol='payment_type_idx',
    handleInvalid='keep'
)
vendor_indexer = StringIndexer(
    inputCol='vendor_str',
    outputCol='vendor_idx',
    handleInvalid='keep'
)

print('StringIndexers defined for payment_type and VendorID')

StringIndexers defined for payment_type and VendorID


In [7]:
# ============================================================
# STEP 5: VECTOR ASSEMBLY + SCALING
# ============================================================

# Feature columns for model
feature_cols = [
    'trip_distance', 'fare_amount', 'pickup_hour',
    'pickup_day', 'pickup_month', 'is_weekend',
    'is_rush_hour', 'trip_duration_min', 'speed_mph',
    'fare_per_mile', 'passenger_count',
    'payment_type_idx', 'vendor_idx'
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features_raw',
    handleInvalid='skip'
)

scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withStd=True,
    withMean=True
)

print('VectorAssembler and StandardScaler defined')
print(f'Feature columns: {feature_cols}')

VectorAssembler and StandardScaler defined
Feature columns: ['trip_distance', 'fare_amount', 'pickup_hour', 'pickup_day', 'pickup_month', 'is_weekend', 'is_rush_hour', 'trip_duration_min', 'speed_mph', 'fare_per_mile', 'passenger_count', 'payment_type_idx', 'vendor_idx']


In [8]:
# ============================================================
# STEP 6: BUILD PYSPARK PIPELINE
# ============================================================

# Define Pipeline stages (without model – model added in Task 3)
preprocessing_pipeline = Pipeline(stages=[
    payment_indexer,
    vendor_indexer,
    assembler,
    scaler
])

print('Pipeline defined with stages:')
print('  1. StringIndexer (payment_type)')
print('  2. StringIndexer (VendorID)')
print('  3. VectorAssembler (13 features -> features_raw)')
print('  4. StandardScaler (features_raw -> features)')

# Filter to valid rows only
df_model = df_feat.select(
    'trip_distance', 'fare_amount', 'pickup_hour',
    'pickup_day', 'pickup_month', 'is_weekend',
    'is_rush_hour', 'trip_duration_min', 'speed_mph',
    'fare_per_mile', 'passenger_count',
    'payment_type_str', 'vendor_str', 'high_tip'
).filter(
    col('trip_duration_min').between(1, 180)
).filter(
    col('speed_mph').between(0, 120)
).filter(
    col('fare_per_mile').between(0, 100)
).na.drop()

print(f'\nFinal modelling dataset: {df_model.count():,} rows')

Pipeline defined with stages:
  1. StringIndexer (payment_type)
  2. StringIndexer (VendorID)
  3. VectorAssembler (13 features -> features_raw)
  4. StandardScaler (features_raw -> features)


[Stage 94:===================================================>    (11 + 1) / 12]


Final modelling dataset: 2,871,351 rows


In [9]:
# ============================================================
# STEP 7: FIT PREPROCESSING PIPELINE & SAVE
# ============================================================

# Fit pipeline
pipeline_model = preprocessing_pipeline.fit(df_model)
df_transformed = pipeline_model.transform(df_model)

print('Pipeline fitted and data transformed')
df_transformed.select('features', 'high_tip').show(5)

# Train/test split
train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=42)
print(f'Training set: {train_df.count():,} rows')
print(f'Test set: {test_df.count():,} rows')

# Save for Task 3
train_df.write.mode('overwrite').parquet('./data/train_processed.parquet')
test_df.write.mode('overwrite').parquet('./data/test_processed.parquet')
print('Saved train and test datasets')

# Screenshot evidence: partition count
print(f'\nData ingestion partition count: {df_model.rdd.getNumPartitions()}')

# spark.stop()

26/06/29 13:49:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Pipeline fitted and data transformed


+--------------------+--------+
|            features|high_tip|
+--------------------+--------+
|[0.56668945221983...|       1|
|[-0.1175188409620...|       1|
|[-0.3659112906320...|       0|
|[-0.1988109153994...|       1|
|[-0.1491324254654...|       0|
+--------------------+--------+
only showing top 5 rows


Training set: 2,297,635 rows


Test set: 573,716 rows


Saved train and test datasets


[Stage 147:==================================================>    (11 + 1) / 12]


Data ingestion partition count: 16


26/06/29 15:36:34 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 141700 ms exceeds timeout 120000 ms
26/06/29 15:36:34 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/29 15:36:37 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o